# End-to-End Usage Example
This notebook demonstrates a full workflow for `intervene_ar`: install/import, data preparation, train/load (Phase 1 + Phase 2), and evaluation.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free,compute_cap --format=csv,noheader

In [ ]:
%pip install -e .

In [ ]:
import importlib
import sys
from pathlib import Path

import joblib
import pandas as pd
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import intervene_ar.config.dataset_config as dataset_config
import intervene_ar.config.model_config as model_config
from intervene_ar import dataset, schedulers, utils, loss, embedder, transformer, inference, diagnose

# Snapshot dynamic keys (computed at runtime, not defined in config files).
_old_model_config = dict(getattr(model_config, "MODEL_CONFIG", {}))
_old_training_settings = dict(getattr(model_config, "TRAINING_SETTINGS", {}))
_old_filter_criteria = getattr(dataset_config, "INCLUSION_EXCLUSION_CRITERIA", None)

for module in (
    dataset_config,
    model_config,
    dataset,
    schedulers,
    utils,
    loss,
    embedder,
    transformer,
    inference,
    diagnose
):
    importlib.reload(module)

# Restore only keys absent from the reloaded file (e.g. ctx_dim set by data cell).
# File-defined keys are intentionally left as reloaded.
for k, v in _old_model_config.items():
    if k not in model_config.MODEL_CONFIG:
        model_config.MODEL_CONFIG[k] = v
for k, v in _old_training_settings.items():
    if k not in model_config.TRAINING_SETTINGS:
        model_config.TRAINING_SETTINGS[k] = v
if _old_filter_criteria is not None and not hasattr(dataset_config, "INCLUSION_EXCLUSION_CRITERIA"):
    dataset_config.INCLUSION_EXCLUSION_CRITERIA = _old_filter_criteria

from intervene_ar.config.model_config import MODEL_CONFIG, TRAINING_SETTINGS

print("Modules reloaded. File-defined config updated; runtime-computed keys preserved.")
print(f"ctx_dim = {MODEL_CONFIG.get('ctx_dim')}  |  embed_dim = {MODEL_CONFIG.get('embed_dim')}")

In [ ]:
# Optional cleanup controls for checkpoint artifacts (useful on restricted VMs)
CLEAN_SCALER = False
CLEAN_TOKENIZER = False
CLEAN_EMBEDDER = False
CLEAN_TRANSFORMER = False

# Convenience switch: if True, cleans all artifacts above
CLEAN_ALL = False

clean_scaler = CLEAN_SCALER or CLEAN_ALL
clean_tokenizer = CLEAN_TOKENIZER or CLEAN_ALL
clean_embedder = CLEAN_EMBEDDER or CLEAN_ALL
clean_transformer = CLEAN_TRANSFORMER or CLEAN_ALL

paths_to_delete = []

if clean_scaler:
    paths_to_delete.append(Path(model_config.CHECKPOINT_PATH) / "scaler.pkl")
if clean_tokenizer:
    paths_to_delete.append(Path(model_config.CHECKPOINT_PATH) / "tokenizer.pt")
if clean_embedder:
    embedder_ckpt = Path(model_config.PHASE1_CHECKPOINT).resolve()
    paths_to_delete.extend([embedder_ckpt, embedder_ckpt.parent / "ckpt_last.pt"])
if clean_transformer:
    transformer_ckpt = Path(model_config.PHASE2_CHECKPOINT).resolve()
    paths_to_delete.extend([transformer_ckpt, transformer_ckpt.parent / "ckpt_last.pt"])
    transformer_ckpt = Path(model_config.PHASE3_CHECKPOINT).resolve()
    paths_to_delete.extend([transformer_ckpt, transformer_ckpt.parent / "ckpt_last.pt"])

if not paths_to_delete:
    print("No cleanup selected. Set CLEAN_* = True (or CLEAN_ALL = True) and re-run this cell.")
else:
    removed, missing, failed = [], [], []
    for path in dict.fromkeys(paths_to_delete):  # dedupe while preserving order
        try:
            if path.exists():
                path.unlink()
                removed.append(path)
            else:
                missing.append(path)
        except Exception as exc:
            failed.append((path, str(exc)))

    print("Checkpoint cleanup summary:")
    print(f"  Removed: {len(removed)}")
    for path in removed:
        print(f"    - {path}")

    print(f"  Not found: {len(missing)}")
    for path in missing:
        print(f"    - {path}")

    if failed:
        print(f"  Failed: {len(failed)}")
        for path, err in failed:
            print(f"    - {path} -> {err}")

## 1) Data Load and Processing

In [ ]:
# Data Source
# DATA_SOURCE_MODE:
#   "pre_split" use TRAIN_* / TEST_* files from dataset_config (recommended for VM runs)
#   "source_split" load from data/source/, split by PatientId, optionally save splits
DATA_SOURCE_MODE = "pre_split"

if DATA_SOURCE_MODE not in {"pre_split", "source_split"}:
    raise ValueError("DATA_SOURCE_MODE must be 'pre_split' or 'source_split'.")

SOURCE_TEMPORAL_DATA_FILE = PROJECT_ROOT / "data" / "source" / "synthetic_diabetes_temporal_data.csv"
SOURCE_CTX_DATA_FILE      = PROJECT_ROOT / "data" / "source" / "synthetic_diabetes_context_data.csv"
SOURCE_TEST_SIZE   = 0.2
SOURCE_SPLIT_SEED  = 42
CONTEXT_COLUMN_RENAME_MAP = {}
CONTEXT_COLUMNS_TO_DROP   = []
SAVE_SOURCE_SPLITS = True
SOURCE_OUTPUT_TRAIN_TEMPORAL_FILE = PROJECT_ROOT / "data" / "train" / "synthetic_diabetes_temporal_data.csv"
SOURCE_OUTPUT_TRAIN_CTX_FILE      = PROJECT_ROOT / "data" / "train" / "synthetic_diabetes_context_data.csv"
SOURCE_OUTPUT_TEST_TEMPORAL_FILE  = PROJECT_ROOT / "data" / "test"  / "synthetic_diabetes_temporal_data.csv"
SOURCE_OUTPUT_TEST_CTX_FILE       = PROJECT_ROOT / "data" / "test"  / "synthetic_diabetes_context_data.csv"

# Train/Val split settings
TRAIN_VAL_SIZE       = 0.2
TRAIN_VAL_SPLIT_SEED = 42

# Optional patient sampling (None = all)
SAMPLE_PATIENTS = None
NUM_WORKERS = 4 if SAMPLE_PATIENTS is None else 0

# Tokenizer
TOKENIZER_PATH     = Path(model_config.CHECKPOINT_PATH) / "tokenizer.pt"
TOKENIZER_FIT_SCOPE = "all_processed"   # "train_only" | "train_val" | "all_processed"

# Misc
AUTO_SET_CTX_DIM = True
K_RANGE          = range(3, 8)
MAX_LEN          = 1000
TEMPERATURE      = 1.0
TIME_BIAS_HOURS  = 48

print(f"Data mode: {DATA_SOURCE_MODE}")

# Load raw data
if DATA_SOURCE_MODE == "pre_split":
    train_temporal_raw = pd.read_csv(dataset_config.TRAIN_TEMPORAL_DATA_FILE, low_memory=False)
    train_ctx_raw      = pd.read_csv(dataset_config.TRAIN_CTX_DATA_FILE)
    eval_temporal_raw  = pd.read_csv(dataset_config.TEST_TEMPORAL_DATA_FILE,  low_memory=False)
    eval_ctx_raw       = pd.read_csv(dataset_config.TEST_CTX_DATA_FILE)
else:
    source_temporal_raw = pd.read_csv(SOURCE_TEMPORAL_DATA_FILE, low_memory=False)
    source_ctx_raw      = pd.read_csv(SOURCE_CTX_DATA_FILE)
    if CONTEXT_COLUMN_RENAME_MAP:
        source_ctx_raw = source_ctx_raw.rename(columns=CONTEXT_COLUMN_RENAME_MAP)
    if CONTEXT_COLUMNS_TO_DROP:
        source_ctx_raw = source_ctx_raw.drop(columns=CONTEXT_COLUMNS_TO_DROP, errors="ignore")
    source_pids = np.sort(source_temporal_raw["PatientId"].dropna().unique())
    train_pool_ids, eval_ids = train_test_split(
        source_pids, test_size=SOURCE_TEST_SIZE, random_state=SOURCE_SPLIT_SEED)
    train_temporal_raw = source_temporal_raw[source_temporal_raw["PatientId"].isin(train_pool_ids)].copy()
    train_ctx_raw      = source_ctx_raw[source_ctx_raw["PatientId"].isin(train_pool_ids)].copy()
    eval_temporal_raw  = source_temporal_raw[source_temporal_raw["PatientId"].isin(eval_ids)].copy()
    eval_ctx_raw       = source_ctx_raw[source_ctx_raw["PatientId"].isin(eval_ids)].copy()
    if SAVE_SOURCE_SPLITS:
        for p in [SOURCE_OUTPUT_TRAIN_TEMPORAL_FILE, SOURCE_OUTPUT_TRAIN_CTX_FILE,
                  SOURCE_OUTPUT_TEST_TEMPORAL_FILE, SOURCE_OUTPUT_TEST_CTX_FILE]:
            p.parent.mkdir(parents=True, exist_ok=True)
        train_temporal_raw.to_csv(SOURCE_OUTPUT_TRAIN_TEMPORAL_FILE, index=False)
        train_ctx_raw.to_csv(SOURCE_OUTPUT_TRAIN_CTX_FILE, index=False)
        eval_temporal_raw.to_csv(SOURCE_OUTPUT_TEST_TEMPORAL_FILE, index=False)
        eval_ctx_raw.to_csv(SOURCE_OUTPUT_TEST_CTX_FILE, index=False)

if SAMPLE_PATIENTS is not None:
    sampled_ids = sorted(random.sample(list(train_temporal_raw["PatientId"].dropna().unique()), SAMPLE_PATIENTS))
    train_temporal_raw = train_temporal_raw[train_temporal_raw["PatientId"].isin(sampled_ids)].copy()
    train_ctx_raw      = train_ctx_raw[train_ctx_raw["PatientId"].isin(sampled_ids)].copy()

# Process full train pool (fits scaler; enables stratified split on processed tokens)
print("Processing full train pool (fits scaler)...")
train_pool_processor = dataset.DataProcessor(
    train_temporal_raw.copy(), train_ctx_raw.copy(),
    scaler=None,
    tak_repo_path=dataset_config.TAK_REPO_PATH,
    checkpoint_path=model_config.CHECKPOINT_PATH,
)
train_pool_df, train_pool_ctx_df = train_pool_processor.run()

scaler_path = Path(model_config.CHECKPOINT_PATH) / "scaler.pkl"
if not scaler_path.exists():
    raise FileNotFoundError(f"Expected scaler at {scaler_path}")
scaler = joblib.load(scaler_path)

# Build / load tokenizer
if TOKENIZER_PATH.exists():
    print(f"Loading tokenizer from: {TOKENIZER_PATH}")
    tokenizer = dataset.EMRTokenizer.load(TOKENIZER_PATH)
else:
    print("Processing eval split for tokenizer fitting...")
    eval_proc_df = dataset.DataProcessor(
        eval_temporal_raw.copy(), eval_ctx_raw.copy(),
        scaler=scaler,
        tak_repo_path=dataset_config.TAK_REPO_PATH,
        checkpoint_path=model_config.CHECKPOINT_PATH,
    ).run()[0]

    if TOKENIZER_FIT_SCOPE == "train_only":
        tokenizer_fit_df = train_pool_df
    elif TOKENIZER_FIT_SCOPE == "train_val":
        tokenizer_fit_df = train_pool_df   # val not yet processed; use full pool as approximation
    else:
        tokenizer_fit_df = pd.concat([train_pool_df, eval_proc_df], ignore_index=True)

    print(f"Building tokenizer ({TOKENIZER_FIT_SCOPE}): "
          f"{tokenizer_fit_df['PatientId'].nunique()} patients, "
          f"{tokenizer_fit_df['PositionToken'].nunique()} unique tokens")
    tokenizer = dataset.EMRTokenizer.from_processed_df(tokenizer_fit_df)
    tokenizer.save(TOKENIZER_PATH)
    print(f"Saved tokenizer to: {TOKENIZER_PATH}")

# Stratified train/val split
# Strata key = each patient's rarest valid outcome (by prevalence ratio).
# Guarantees at least one representative of each kept outcome in both train and val.
train_pool_pids = np.sort(train_pool_df["PatientId"].dropna().unique())
patient_tokens_map = train_pool_df.groupby("PatientId")["PositionToken"].apply(set)
strat_labels = []
for pid in train_pool_pids:
    patient_outcomes = [n for n in dataset_config.OUTCOMES if n in patient_tokens_map.get(pid, set())]
    if patient_outcomes and tokenizer.outcome_patient_ratios:
        rarest = min(patient_outcomes, key=lambda n: tokenizer.outcome_patient_ratios.get(n, 1.0))
        strat_labels.append(rarest)
    else:
        strat_labels.append("__common__")

train_ids, val_ids = train_test_split(
    train_pool_pids, test_size=TRAIN_VAL_SIZE, random_state=TRAIN_VAL_SPLIT_SEED,
    stratify=strat_labels,
)

# Slice already-processed data; reprocess val with fitted scaler
train_temporal_df = train_pool_df[train_pool_df["PatientId"].isin(train_ids)].copy()
train_ctx_df      = train_pool_ctx_df[train_pool_ctx_df.index.isin(train_ids)].copy()

val_temporal_raw  = train_temporal_raw[train_temporal_raw["PatientId"].isin(val_ids)].copy()
val_ctx_raw       = train_ctx_raw[train_ctx_raw["PatientId"].isin(val_ids)].copy()
print("Processing val split (uses fitted scaler)...")
val_temporal_df, val_ctx_df = dataset.DataProcessor(
    val_temporal_raw, val_ctx_raw,
    scaler=scaler,
    tak_repo_path=dataset_config.TAK_REPO_PATH,
    checkpoint_path=model_config.CHECKPOINT_PATH,
).run()

print("Processing eval split (uses fitted scaler)...")
eval_temporal_df, eval_ctx_df = dataset.DataProcessor(
    eval_temporal_raw.copy(), eval_ctx_raw.copy(),
    scaler=scaler,
    tak_repo_path=dataset_config.TAK_REPO_PATH,
    checkpoint_path=model_config.CHECKPOINT_PATH,
).run()

# Build datasets
train_ds = dataset.EMRDataset(train_temporal_df, train_ctx_df, tokenizer=tokenizer)
val_ds   = dataset.EMRDataset(val_temporal_df,   val_ctx_df,   tokenizer=tokenizer)
eval_ds  = dataset.EMRDataset(eval_temporal_df,  eval_ctx_df,  tokenizer=tokenizer)

if AUTO_SET_CTX_DIM:
    model_config.MODEL_CONFIG["ctx_dim"] = int(train_ds.context_df.shape[1])
    print(f"Updated MODEL_CONFIG['ctx_dim'] -> {model_config.MODEL_CONFIG['ctx_dim']}")

print(f"Train patients: {len(train_ids)} | Val patients: {len(val_ids)} | Eval patients: {eval_temporal_df['PatientId'].nunique()}")
print(f"Train records: {len(train_ds.tokens_df):,} | Val records: {len(val_ds.tokens_df):,} | Eval records: {len(eval_ds.tokens_df):,}")


## 2) Training

In [ ]:
# Training Control Panel
#
# RUN_PHASE1 : Whether to run Phase-1 (embedder) training.
#              True  -> train the embedder (fresh or resumed per RESUME_TRAINING).
#              False -> skip; best available Phase-1 checkpoint is loaded instead.
#
# RUN_PHASE2 : Whether to run Phase-2 (transformer) training.
#              True  -> train the transformer (fresh or resumed per RESUME_TRAINING).
#              False -> skip; best available Phase-2 checkpoint is loaded instead.
#
# RUN_PHASE3 : Whether to run Phase-3 (outcome head alignment).
#              True  -> fine-tune outcome_head on free-running trajectories.
#              False -> skip; loads Phase-3 checkpoint if available, otherwise uses
#                      Phase-2 outcome head (may have teacher-forcing distribution gap).
#
# RESUME_TRAINING : Whether each active phase continues from its last checkpoint.
#              True  -> restore weights + optimizer + scheduler + aux-scheduler and continue.
#              False -> start that phase from epoch 0 with fresh optimizer/scheduler state.
#              (Has no effect on phases that are skipped -- those always load best checkpoint.)
#
#
# Common configurations:
#   Full fresh start             RUN_PHASE1=True,  RUN_PHASE2=True,  RUN_PHASE3=True,  RESUME_TRAINING=False
#   Resume all phases            RUN_PHASE1=True,  RUN_PHASE2=True,  RUN_PHASE3=True,  RESUME_TRAINING=True
#   Skip phase 1, fresh ph.2-3   RUN_PHASE1=False, RUN_PHASE2=True,  RUN_PHASE3=True,  RESUME_TRAINING=False
#   Phase 3 only (post-training) RUN_PHASE1=False, RUN_PHASE2=False, RUN_PHASE3=True,  RESUME_TRAINING=True
#   Evaluate only (no training)  RUN_PHASE1=False, RUN_PHASE2=False, RUN_PHASE3=False, RESUME_TRAINING=False

RUN_PHASE1      = True
RUN_PHASE2      = True
RUN_PHASE3      = True
RESUME_TRAINING = True

print(f"RUN_PHASE1={RUN_PHASE1}  |  RUN_PHASE2={RUN_PHASE2}  |  RUN_PHASE3={RUN_PHASE3}  |  RESUME_TRAINING={RESUME_TRAINING}")

In [ ]:
# DataLoaders
# All DataLoader construction lives here so phases can be re-run independently
# by re-executing only this cell and the training cell below.
#
# Phase 2 uses oversample=True + bucket_batching=True (WeightedBucketBatchSampler):
#   draws indices with rare-outcome weights, then sorts by length to minimise padding.
#   pos_weight is omitted from OutcomeCriterion  the sampler already rebalances.
# Phases 1 and 3 share train_dl (natural distribution, bucket-batched).
#
# NUM_WORKERS is set to 0 when SAMPLE_PATIENTS is not None (Windows multiprocessing
# cannot spawn worker processes inside a Jupyter kernel with a small dataset).
bs = model_config.TRAINING_SETTINGS["batch_size"]

train_dl = dataset.get_dataloader(
    train_ds, batch_size=bs, collate_fn=dataset.collate_emr,
    oversample=False, bucket_batching=True, num_workers=NUM_WORKERS,
)
oversampled_train_dl = dataset.get_dataloader(
    train_ds, batch_size=bs, collate_fn=dataset.collate_emr,
    oversample=True, bucket_batching=True, num_workers=NUM_WORKERS,
)
val_dl = dataset.get_dataloader(
    val_ds, batch_size=bs, collate_fn=dataset.collate_emr,
    oversample=False, bucket_batching=True, num_workers=NUM_WORKERS,
)

print(f"DataLoaders ready  batch_size={bs}, num_workers={NUM_WORKERS}")
print(f"  train_dl             : {len(train_dl)} batches (bucket, no oversample)")
print(f"  oversampled_train_dl : {len(oversampled_train_dl)} batches (weighted bucket)")
print(f"  val_dl               : {len(val_dl)} batches (bucket, no oversample)")

In [ ]:
def _load_embedder_checkpoint(best_path, tokenizer):
    """Load best Phase-1 checkpoint, falling back to last if best is not yet saved."""
    ckpt_best = Path(best_path).resolve()
    ckpt_last = ckpt_best.parent / "ckpt_last.pt"
    ckpt = ckpt_best if ckpt_best.exists() else ckpt_last
    if not ckpt.exists():
        raise FileNotFoundError(
            "No Phase-1 checkpoint found. Run with RUN_PHASE1=True first."
        )
    print(f"Loading embedder from: {ckpt}")
    model, *_ = embedder.EMREmbedding.load(ckpt, tokenizer=tokenizer)
    return model

def _load_transformer_checkpoint(best_path, embedder_model):
    """Load best Phase-2 checkpoint, falling back to last if best is not yet saved."""
    ckpt_best = Path(best_path).resolve()
    ckpt_last = ckpt_best.parent / "ckpt_last.pt"
    ckpt = ckpt_best if ckpt_best.exists() else ckpt_last
    if not ckpt.exists():
        raise FileNotFoundError(
            "No Phase-2 checkpoint found. Run with RUN_PHASE2=True first."
        )
    print(f"Loading transformer from: {ckpt}")
    model, *_ = transformer.InterveneGPT.load(ckpt, embedder=embedder_model)
    return model


# Phase 1: embedder
# RUN_PHASE1=True : always start with a fresh embedder instance.
#   train_embedder() handles checkpoint loading internally when RESUME_TRAINING=True,
#   restoring weights, optimizer, scheduler, and aux-scheduler state.
# RUN_PHASE1=False: load the best available Phase-1 checkpoint directly.

if RUN_PHASE1:
    embedder_model = embedder.EMREmbedding(
        tokenizer=tokenizer,
        ctx_dim=model_config.MODEL_CONFIG.get("ctx_dim"),
        time2vec_dim=model_config.MODEL_CONFIG.get("time2vec_dim"),
        embed_dim=model_config.MODEL_CONFIG.get("embed_dim"),
    )
    embedder_model, _, _ = embedder.train_embedder(
        embedder=embedder_model,
        train_loader=train_dl,
        val_loader=val_dl,
        resume=RESUME_TRAINING,
        checkpoint_path=model_config.PHASE1_CHECKPOINT,
        training_settings=model_config.TRAINING_SETTINGS,
    )
else:
    embedder_model = _load_embedder_checkpoint(model_config.PHASE1_CHECKPOINT, tokenizer)


# Phase 2: transformer
# RUN_PHASE2=True : always initialize a fresh InterveneGPT instance wrapping the embedder above.
#   pretrain_transformer() handles checkpoint loading internally when RESUME_TRAINING=True.
#   The fresh InterveneGPT passed in is discarded on resume -- only its architecture is used.
# RUN_PHASE2=False: load the best available Phase-2 checkpoint directly.

if RUN_PHASE2:
    model = transformer.InterveneGPT(cfg=model_config.MODEL_CONFIG, embedder=embedder_model)
    model, _, _ = transformer.pretrain_transformer(
        model=model,
        train_dl=oversampled_train_dl,
        val_dl=val_dl,
        resume=RESUME_TRAINING,
        checkpoint_path=model_config.PHASE2_CHECKPOINT,
        training_settings=model_config.TRAINING_SETTINGS,
    )
else:
    model = _load_transformer_checkpoint(model_config.PHASE2_CHECKPOINT, embedder_model)


# Phase 3: outcome head fine-tuning
# Uses natural-distribution batches (oversample=False) so pos_weight in
# BCEWithLogitsLoss correctly compensates for class imbalance without
# double-counting (unlike Phase-2 which uses oversample=True).
# RUN_PHASE3=False: loads Phase-3 checkpoint if available (full InterveneGPT.load()),
#   otherwise the Phase-2 outcome head is used as-is.

if RUN_PHASE3:
    model, _, _ = transformer.finetune_transformer(
        model=model,
        train_dl=train_dl,
        val_dl=val_dl,
        resume=RESUME_TRAINING,
        checkpoint_path=model_config.PHASE3_CHECKPOINT,
        training_settings=model_config.TRAINING_SETTINGS,
    )
else:
    p3_ckpt = Path(model_config.PHASE3_CHECKPOINT)
    p3_last = p3_ckpt.parent / "ckpt_last.pt"
    p3_path = p3_last if p3_last.exists() else (p3_ckpt if p3_ckpt.exists() else None)
    if p3_path:
        print(f"Loading Phase-3 checkpoint from: {p3_path}")
        model, *_ = transformer.InterveneGPT.load(p3_path, embedder=embedder_model)
    else:
        print("No Phase-3 checkpoint found; using outcome head from Phase-2.")

model.eval()
print("Model ready for evaluation.")

## 3) Risk-Based Complication Prediction

Generates a single trajectory per patient (K=1, autoregressive) and reads the outcome head at every step.
This produces a **risk curve** per complication over predicted time.

Evaluation uses **time-stratified AUC**: at each 24 h window the max predicted probability within that
window is compared against whether the complication occurs within that **same 24 h window** in the ground truth.
AUROC and AUPRC are computed per complication then averaged across time windows.


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score
from tqdm.auto import tqdm
from pathlib import Path

from intervene_ar.inference import generate
from intervene_ar.config import dataset_config

# Load best model (safe to re-run independently after a kernel restart)
# Prefers Phase-3 best -> Phase-3 last -> Phase-2 best -> Phase-2 last.
# Requires `tokenizer` to be in scope (run the data cell first).

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def _pick_ckpt(*candidates):
    """Return the first existing path among candidates, or None."""
    for p in candidates:
        if Path(p).exists():
            return Path(p)
    return None

_emb_ckpt = _pick_ckpt(model_config.PHASE1_CHECKPOINT,
                        Path(model_config.PHASE1_CHECKPOINT).parent / "ckpt_last.pt")
if _emb_ckpt is None:
    raise FileNotFoundError("No Phase-1 checkpoint found -- run Phase-1 training first.")

_p3_best = Path(model_config.PHASE3_CHECKPOINT)
_p3_last = _p3_best.parent / "ckpt_last.pt"
_p2_best = Path(model_config.PHASE2_CHECKPOINT)
_p2_last = _p2_best.parent / "ckpt_last.pt"
_tfm_ckpt = _pick_ckpt(_p3_best, _p3_last, _p2_best, _p2_last)
if _tfm_ckpt is None:
    raise FileNotFoundError("No Phase-2 or Phase-3 checkpoint found -- run training first.")

print(f"Loading embedder  : {_emb_ckpt}")
_embedder_model, *_ = embedder.EMREmbedding.load(_emb_ckpt, tokenizer=tokenizer,
                                                  map_location=DEVICE)

print(f"Loading transformer: {_tfm_ckpt}")
model, *_ = transformer.InterveneGPT.load(_tfm_ckpt, embedder=_embedder_model,
                                  map_location=DEVICE)
model = model.cuda()
model.eval()
print(f"Model ready on {DEVICE}.")

OUTCOME_NAMES = model.outcome_names   # list of outcome token strings
P_COLS        = [f'P_{n}' for n in OUTCOME_NAMES]

### 3a) Trained Model Debugging Block

In [ ]:
from intervene_ar.diagnose import (
    probe_legality_starvation,
    probe_terminal_logits,
    probe_dt_head,
    probe_dt_components,
    probe_outcome_label_alignment,
    probe_calibration_by_abs_time,
    probe_outcome_logit_distribution,
    probe_outcome_lm_coupling,
)

probe_legality_starvation(model, val_dl, tokenizer)            # confirms terminals aren't masked at GT positions
probe_terminal_logits(model, val_dl, tokenizer, n_batches=2)   # explains "no natural stop"
probe_dt_head(model, val_dl, n_batches=2)                       # detects Δt collapse
probe_dt_components(model, val_dl, n_batches=2)                  # inspects Δt components
probe_outcome_label_alignment(model, val_dl, tokenizer, n_batches=2)  # finds RELEASE flip
probe_calibration_by_abs_time(model, val_dl, tokenizer, n_batches=2)  # Time2Vec OOD
probe_outcome_logit_distribution(model, val_dl, n_batches=2)   # confirms over-sharp logits
probe_outcome_lm_coupling(model, val_dl, n_batches=2)  # inspects LM coupling

### 3b) Ground Truth Extraction

In [ ]:
# ---------------------------------------------------------------------------
# Evaluation constants (locked, matching the headline metrics)
# ---------------------------------------------------------------------------
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve
from intervene_ar.config.dataset_config import OUTCOME_RARE_THRESHOLD_PCT

EVAL_INPUT_DAYS         = 2      # days of patient history used as generation seed
EVAL_WINDOW_HOURS       = 24.0   # non-overlapping prediction window size
EVAL_GRACE_HOURS        = 24.0   # tolerance added to each window edge for positive labelling
EVAL_MAX_LEN            = 500    # max generated steps per patient
EVAL_TEMPERATURE        = 1.0    # sampling temperature
EVAL_FULL_HORIZON_HOURS = 336.0  # cap per-patient eval horizon at 14 days

# Prevalence floor for an outcome's AUC to be emitted (same 1% used at data-load time).
EVAL_PREVALENCE_THRESHOLD = OUTCOME_RARE_THRESHOLD_PCT / 100.0

# RELEASE_EVENT is the negation of DEATH in this cohort; including both in the
# AUC headline double-counts the same terminal-event ranking task. RELEASE stays
# in the LM vocab (model emits it -> trajectory-length signal) and is reported
# via length-of-stay MAE instead.
AUC_EXCLUDE = ("RELEASE_EVENT",)


def _min_positives(n_patients, threshold=EVAL_PREVALENCE_THRESHOLD):
    return max(1, int(round(threshold * n_patients)))


# ---------------------------------------------------------------------------
# Ground-truth extraction
# ---------------------------------------------------------------------------
def extract_ground_truth(eval_ds, outcome_names):
    """{pid: {outcome: first_time_hours or np.inf}} from full (untruncated) sequences."""
    outcome_set = set(outcome_names)
    tok_col = "PositionToken" if "PositionToken" in next(iter(eval_ds.patient_groups.values())).columns else "Token"
    gt = {}
    for pid in eval_ds.patient_ids:
        df = eval_ds.patient_groups[pid]
        patient_gt = {n: np.inf for n in outcome_names}
        for _, row in df.iterrows():
            tok = row[tok_col]
            if tok in outcome_set:
                t = row["TimePoint"]
                if t < patient_gt[tok]:
                    patient_gt[tok] = t
        gt[pid] = patient_gt
    return gt


def extract_ground_truth_episodes(eval_ds, outcome_names):
    """{pid: {outcome: [t1, t2, ...]}} all occurrence times."""
    outcome_set = set(outcome_names)
    tok_col = "PositionToken" if "PositionToken" in next(iter(eval_ds.patient_groups.values())).columns else "Token"
    gt = {}
    for pid in eval_ds.patient_ids:
        df = eval_ds.patient_groups[pid]
        patient_gt = {n: [] for n in outcome_names}
        for _, row in df.iterrows():
            tok = row[tok_col]
            if tok in outcome_set:
                patient_gt[tok].append(row["TimePoint"])
        gt[pid] = patient_gt
    return gt


def extract_patient_horizons(eval_ds, full_horizon_hours=EVAL_FULL_HORIZON_HOURS):
    """Per-patient evaluation horizon = min(last GT event time, full_horizon_hours)."""
    out = {}
    for pid in eval_ds.patient_ids:
        df = eval_ds.patient_groups[pid]
        last_t = float(df["TimePoint"].max()) if len(df) else 0.0
        out[pid] = min(last_t, full_horizon_hours)
    return out


gt_first         = extract_ground_truth(eval_ds, OUTCOME_NAMES)
gt_episodes      = extract_ground_truth_episodes(eval_ds, OUTCOME_NAMES)
patient_horizons = extract_patient_horizons(eval_ds)

print(f"Ground truth extracted for {len(gt_first)} patients.")
horiz = np.array(list(patient_horizons.values()), dtype=float)
print(f"Horizons (h): median={np.median(horiz):.1f}, mean={horiz.mean():.1f}, "
      f"p90={np.percentile(horiz, 90):.1f}, max={horiz.max():.1f}")

# Aliases used by the calibration cell below.
gt_labels          = gt_first
gt_labels_episodes = gt_episodes

summary = {n: sum(1 for p in gt_first if gt_first[p][n] < np.inf) for n in OUTCOME_NAMES}
pd.Series(summary, name="patients_with_complication").sort_values(ascending=False)


### 3c) Generate Risk Curves

In [ ]:
# Truncated eval dataset (EVAL_INPUT_DAYS-day seed) + autoregressive generation.
print(f"Building truncated eval dataset ({EVAL_INPUT_DAYS}-day input window)...")
eval_input_processor = dataset.DataProcessor(
    eval_temporal_raw.copy(), eval_ctx_raw.copy(),
    scaler=scaler,
    tak_repo_path=dataset_config.TAK_REPO_PATH,
    checkpoint_path=model_config.CHECKPOINT_PATH,
    max_input_days=EVAL_INPUT_DAYS,
)
eval_input_temporal_df, eval_input_ctx_df = eval_input_processor.run()
eval_ds_input = dataset.EMRDataset(eval_input_temporal_df, eval_input_ctx_df, tokenizer=tokenizer)

risk_df = generate(
    model, eval_ds_input,
    max_len=EVAL_MAX_LEN,
    temperature=EVAL_TEMPERATURE,
    top_k=None,
    rep_decay=0.6,
    collect_risk_scores=True,
)
print(f'Risk curves: {len(risk_df)} rows, {risk_df["PatientId"].nunique()} patients.')
risk_df.head()


### 3d) Headline Metrics — patient-level peak-detector AUC


In [ ]:
# === HEADLINE: patient-level peak-detector AUC + F1 ============================
# For each (patient, outcome): score = max P_outcome(t) over all generated positions,
# label = 1 iff the outcome occurred at any point in the GT trajectory.
# AUROC/AUPRC pooled once per outcome -> stable on rare outcomes.
# RELEASE_EVENT excluded (negation of DEATH in this cohort).
def per_patient_max_auc(risk_df, gt_episodes, outcome_names, min_positives=None):
    gen_df   = risk_df[risk_df["IsInput"] == 0]
    p_cols   = [f"P_{n}" for n in outcome_names]
    all_pids = list(risk_df["PatientId"].unique())
    n_patients = len(all_pids)
    if min_positives is None:
        min_positives = _min_positives(n_patients)

    max_per_patient = {pid: {c: 0.0 for c in p_cols} for pid in all_pids}
    if len(gen_df):
        grouped = gen_df.groupby("PatientId")[p_cols].max()
        for pid, row in grouped.iterrows():
            for c in p_cols:
                max_per_patient[pid][c] = float(row[c])

    rows = []
    for name in outcome_names:
        pcol = f"P_{name}"
        scores, labels = [], []
        for pid in all_pids:
            scores.append(max_per_patient[pid][pcol])
            labels.append(int(len(gt_episodes.get(pid, {}).get(name, [])) > 0))
        labels = np.array(labels); scores = np.array(scores)
        n_pos = int(labels.sum()); n_neg = int((1 - labels).sum())
        prevalence = n_pos / max(1, n_pos + n_neg)

        if n_pos < min_positives or n_neg < min_positives:
            rows.append({"outcome": name, "auroc": np.nan, "auprc": np.nan,
                         "max_f1": np.nan, "max_f1_threshold": np.nan, "f1_at_0_5": np.nan,
                         "n_pos": n_pos, "n_neg": n_neg, "prevalence": prevalence})
            continue

        precisions, recalls, thresholds = precision_recall_curve(labels, scores)
        f1s = np.where((precisions + recalls) > 0,
                       2 * precisions * recalls / np.maximum(precisions + recalls, 1e-12), 0.0)
        best_idx = int(np.argmax(f1s))
        max_f1 = float(f1s[best_idx])
        max_f1_thr = float(thresholds[best_idx]) if best_idx < len(thresholds) else (
            float(thresholds[-1]) if len(thresholds) else 0.5)

        preds_05 = (scores >= 0.5).astype(int)
        tp = int(((preds_05 == 1) & (labels == 1)).sum())
        fp = int(((preds_05 == 1) & (labels == 0)).sum())
        fn = int(((preds_05 == 0) & (labels == 1)).sum())
        prec_05 = tp / max(tp + fp, 1); rec_05 = tp / max(tp + fn, 1)
        f1_at_0_5 = 2 * prec_05 * rec_05 / (prec_05 + rec_05) if (prec_05 + rec_05) > 0 else 0.0

        rows.append({
            "outcome": name,
            "auroc": float(roc_auc_score(labels, scores)),
            "auprc": float(average_precision_score(labels, scores)),
            "max_f1": max_f1, "max_f1_threshold": max_f1_thr, "f1_at_0_5": float(f1_at_0_5),
            "n_pos": n_pos, "n_neg": n_neg, "prevalence": prevalence,
        })
    return pd.DataFrame(rows).set_index("outcome").sort_values("auroc", ascending=False)


def weighted_mean_auc(auc_table, by="n_pos"):
    """Support-weighted mean of AUROC / AUPRC / F1 across outcomes with non-NaN AUC."""
    tbl = auc_table.dropna(subset=["auroc"])
    if len(tbl) == 0:
        nan = float("nan")
        return {"auroc_weighted": nan, "auprc_weighted": nan,
                "auroc_simple": nan, "auprc_simple": nan,
                "max_f1_weighted": nan, "max_f1_simple": nan,
                "f1_at_0_5_weighted": nan, "f1_at_0_5_simple": nan,
                "n_outcomes_used": 0}
    w = tbl[by].astype(float).values
    w = w / w.sum() if w.sum() > 0 else np.ones_like(w) / len(w)
    return {
        "auroc_weighted":     float((tbl["auroc"].values * w).sum()),
        "auprc_weighted":     float((tbl["auprc"].values * w).sum()),
        "auroc_simple":       float(tbl["auroc"].mean()),
        "auprc_simple":       float(tbl["auprc"].mean()),
        "max_f1_weighted":    float((tbl["max_f1"].values    * w).sum()),
        "max_f1_simple":      float(tbl["max_f1"].mean()),
        "f1_at_0_5_weighted": float((tbl["f1_at_0_5"].values * w).sum()),
        "f1_at_0_5_simple":   float(tbl["f1_at_0_5"].mean()),
        "n_outcomes_used":    int(len(tbl)),
    }


auc_outcome_names = [n for n in OUTCOME_NAMES if n not in AUC_EXCLUDE]
print(f"AUC computed over {len(auc_outcome_names)} outcomes (excluded from headline: {list(AUC_EXCLUDE)}).")

patient_auc_table = per_patient_max_auc(risk_df, gt_episodes, auc_outcome_names)
patient_mean      = weighted_mean_auc(patient_auc_table, by="n_pos")

print(f"\n[Patient AUC headline]")
print(f"  auroc_weighted  = {patient_mean['auroc_weighted']:.4f}")
print(f"  auprc_weighted  = {patient_mean['auprc_weighted']:.4f}")
print(f"  max_f1_weighted = {patient_mean['max_f1_weighted']:.4f}")
print(f"  f1@0.5_weighted = {patient_mean['f1_at_0_5_weighted']:.4f}")
print(f"  n_outcomes_used = {patient_mean['n_outcomes_used']}")
patient_auc_table.round(4)


### 3e) Peak-time MAE (nearest GT), Length-of-Stay, generation stats


In [ ]:
# Peak-time MAE to the NEAREST ground-truth occurrence (fair when complications recur).
def time_accuracy_nearest(risk_df, gt_episodes, outcome_names):
    gen_df = risk_df[risk_df["IsInput"] == 0].copy()
    if len(gen_df) == 0:
        return pd.DataFrame()
    p_cols = [f"P_{n}" for n in outcome_names]
    idxmax = gen_df.groupby("PatientId")[p_cols].idxmax()
    rows = []
    for name in outcome_names:
        pcol = f"P_{name}"
        pred_t = gen_df.loc[idxmax[pcol].dropna().astype(int), ["PatientId", "TimePoint"]]
        pred_t = pred_t.set_index("PatientId")["TimePoint"]
        errors = []
        for pid, pt in pred_t.items():
            episodes = gt_episodes.get(pid, {}).get(name, [])
            if not episodes:
                continue
            errors.append(min(abs(pt - t_gt) for t_gt in episodes))
        rows.append({"outcome": name,
                     "mae_hours": float(np.mean(errors)) if errors else np.nan,
                     "n_patients": len(errors)})
    return pd.DataFrame(rows).set_index("outcome").sort_values("mae_hours")


# Length-of-stay regression — replaces RELEASE peak-MAE with a trajectory-length
# regression (admission -> end of input+generated trajectory vs GT release time).
def length_of_stay_mae(risk_df, gt_episodes, release_token="RELEASE_EVENT"):
    errors, gt_vals, pred_vals = [], [], []
    for pid, sub in risk_df.groupby("PatientId"):
        gt_releases = gt_episodes.get(pid, {}).get(release_token, [])
        if not gt_releases:
            continue
        gt_los   = float(min(gt_releases))
        pred_los = float(sub["TimePoint"].max())
        errors.append(abs(pred_los - gt_los))
        gt_vals.append(gt_los); pred_vals.append(pred_los)
    if not errors:
        return {"mae_hours": float("nan"), "median_hours": float("nan"),
                "p90_hours": float("nan"), "n_patients": 0,
                "gt_mean_hours": float("nan"), "pred_mean_hours": float("nan")}
    errs = np.asarray(errors)
    return {"mae_hours": float(errs.mean()),
            "median_hours": float(np.median(errs)),
            "p90_hours": float(np.percentile(errs, 90)),
            "n_patients": int(len(errs)),
            "gt_mean_hours": float(np.mean(gt_vals)),
            "pred_mean_hours": float(np.mean(pred_vals))}


# Trajectory-collapse failure-mode diagnostics — span vs GT horizon ratio.
def compute_gen_stats(risk_df, patient_horizons=None):
    stats = {"gen_n_patients": int(risk_df["PatientId"].nunique())}
    gen_df = risk_df[risk_df["IsInput"] == 0]
    if len(gen_df) == 0:
        return stats
    per_pat_steps = gen_df.groupby("PatientId").size()
    span     = gen_df.groupby("PatientId")["TimePoint"].max() - gen_df.groupby("PatientId")["TimePoint"].min()
    seed_end = gen_df.groupby("PatientId")["TimePoint"].min()

    term_df = gen_df[gen_df["IsTerminal"] == 1]
    if len(term_df):
        term = term_df.groupby("PatientId")["TimePoint"].min()
        within24 = (term - seed_end.loc[term.index]).lt(24.0)
        n_terminal = int(len(term)); frac_early = float(within24.mean())
    else:
        n_terminal = 0; frac_early = 0.0

    stats.update({
        "gen_median_steps": float(per_pat_steps.median()),
        "gen_p90_steps":    float(per_pat_steps.quantile(0.9)),
        "gen_median_hours": float(span.median()),
        "gen_mean_hours":   float(span.mean()),
        "gen_p90_hours":    float(span.quantile(0.9)),
        "gen_max_hours":    float(span.max()),
        "gen_n_with_terminal":        n_terminal,
        "gen_frac_terminal_first24h": frac_early,
    })

    if patient_horizons:
        diffs, gt_spans = [], []
        for pid, s in span.items():
            if pid not in patient_horizons:
                continue
            gt_span = max(0.0, patient_horizons[pid] - seed_end.loc[pid])
            gt_spans.append(gt_span); diffs.append(abs(float(s) - gt_span))
        if diffs:
            stats["gen_length_mae_hrs"] = float(np.mean(diffs))
        if gt_spans:
            gt_arr = np.asarray(gt_spans, dtype=float)
            stats["gt_median_hours"] = float(np.median(gt_arr))
            stats["gt_mean_hours"]   = float(gt_arr.mean())
            stats["gen_to_gt_ratio_median"] = (float(span.median()) / stats["gt_median_hours"]) if stats["gt_median_hours"] > 0 else 0.0
            stats["gen_to_gt_ratio_mean"]   = (float(span.mean())   / stats["gt_mean_hours"])   if stats["gt_mean_hours"]   > 0 else 0.0
    return stats


peak_mae_table = time_accuracy_nearest(risk_df, gt_episodes, auc_outcome_names)
print("Peak-time MAE per outcome (hours):")
display(peak_mae_table.round(2))

los_stats = length_of_stay_mae(risk_df, gt_episodes)
print(f"\nLength-of-stay MAE: {los_stats['mae_hours']:.1f}h  "
      f"(median {los_stats['median_hours']:.1f}h, p90 {los_stats['p90_hours']:.1f}h, n={los_stats['n_patients']})")
print(f"  GT mean = {los_stats['gt_mean_hours']:.1f}h, pred mean = {los_stats['pred_mean_hours']:.1f}h")

gen_stats = compute_gen_stats(risk_df, patient_horizons=patient_horizons)
print("\nGeneration stats:")
for k, v in gen_stats.items():
    print(f"  {k:<30s} = {v}")


### 3e) Temperature Scaling Calibration (local, from pod-exported CSVs)

The pod exports per-(patient, outcome) **peak** logits and binary labels for the
held-out **val** and **test** splits, plus per-step risk curves for a small
exemplar subset of test patients. Everything below is computed locally; no
model forward pass is needed in this notebook.

We fit one temperature scalar `T_k` per outcome on `peaks_val.csv` (LBFGS
minimising BCE of `sigmoid(z/T_k)` against `patient_label`) and apply the frozen
`T_k` to `peaks_test.csv` for out-of-sample reporting. Rank metrics (AUROC,
AUPRC) are unchanged by `T`-scaling and serve as a sanity check.

Expected CSV location (override `CALIB_DIR` below if you've put them elsewhere):

```
results/calibration/peaks_val.csv
results/calibration/peaks_test.csv
results/calibration/risk_curves_test_subset.csv
results/calibration/gt_events_test_subset.csv
```


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from pathlib import Path

# --- Locate exported CSVs (override CALIB_DIR if needed) ---
PROJECT_ROOT = Path.cwd().resolve()
CALIB_DIR    = PROJECT_ROOT / "results" / "calibration"
IMAGES_DIR   = PROJECT_ROOT / "images"
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

peaks_val  = pd.read_csv(CALIB_DIR / "peaks_val.csv")
peaks_test = pd.read_csv(CALIB_DIR / "peaks_test.csv")
print(f"[load] peaks_val: {len(peaks_val):>6} rows / {peaks_val['PatientId'].nunique():>5} patients")
print(f"[load] peaks_test:{len(peaks_test):>6} rows / {peaks_test['PatientId'].nunique():>5} patients")

# Outcomes to calibrate (derived from data — uses whatever is in the CSV)
OUTCOMES_CAL = sorted(peaks_val["outcome"].unique().tolist())
print(f"[load] outcomes ({len(OUTCOMES_CAL)}): {OUTCOMES_CAL}")


# --- Helpers ---
def _sigmoid(z):
    """Stable sigmoid on numpy."""
    z = np.asarray(z, dtype=np.float64)
    return np.where(z >= 0, 1.0 / (1.0 + np.exp(-z)), np.exp(z) / (1.0 + np.exp(z)))


def _ece(probs, labels, n_bins=15):
    """Expected Calibration Error: weighted |conf - acc| over n_bins equi-width bins."""
    probs = np.asarray(probs); labels = np.asarray(labels)
    bin_edges = np.linspace(0, 1, n_bins + 1)
    N = len(probs); e = 0.0
    for k, (lo, hi) in enumerate(zip(bin_edges[:-1], bin_edges[1:])):
        upper = (probs <= hi) if k == n_bins - 1 else (probs < hi)
        mask = (probs >= lo) & upper
        if mask.sum() == 0:
            continue
        e += (mask.sum() / N) * abs(labels[mask].mean() - probs[mask].mean())
    return float(e)


def _fit_T(logits, labels, n_iter=200, lr=0.05):
    """Per-outcome T fit by LBFGS on BCE of sigmoid(z/T) vs labels. Returns float T."""
    labels = np.asarray(labels, dtype=np.float32)
    if labels.sum() < 2 or labels.sum() > len(labels) - 2:
        return 1.0  # degenerate single-class slice
    logits_t = torch.tensor(np.asarray(logits, dtype=np.float32))
    labels_t = torch.tensor(labels)
    T = nn.Parameter(torch.ones(1))
    opt = torch.optim.LBFGS([T], lr=lr, max_iter=n_iter)
    def closure():
        opt.zero_grad()
        loss = nn.functional.binary_cross_entropy_with_logits(logits_t / T, labels_t)
        loss.backward()
        return loss
    opt.step(closure)
    return float(T.item())


# --- Fit T_k per outcome on val ---
temperatures = {}
for name in OUTCOMES_CAL:
    sub = peaks_val[peaks_val["outcome"] == name]
    T = _fit_T(sub["peak_logit"].values, sub["patient_label"].values)
    temperatures[name] = T

print("\nPer-outcome temperature T (fit on VAL; T>1 -> over-confident, T<1 -> under-confident):")
display(pd.Series(temperatures, name="temperature_T").sort_values().to_frame())


# --- Apply T to TEST and compare uncalibrated / calibrated ---
rows, cal_data = [], {}
for name in OUTCOMES_CAL:
    sub = peaks_test[peaks_test["outcome"] == name]
    z = sub["peak_logit"].values.astype(np.float64)
    y = sub["patient_label"].values.astype(np.float64)
    p_uncal = _sigmoid(z)
    p_cal   = _sigmoid(z / temperatures[name])
    cal_data[name] = (z, y)

    def _nll(p, y, eps=1e-7):
        p = np.clip(p, eps, 1 - eps)
        return float(-(y * np.log(p) + (1 - y) * np.log(1 - p)).mean())

    rows.append({
        "outcome":   name,
        "T_k":       temperatures[name],
        "ECE_uncal": _ece(p_uncal, y, n_bins=15),
        "ECE_cal":   _ece(p_cal,   y, n_bins=15),
        "NLL_uncal": _nll(p_uncal, y),
        "NLL_cal":   _nll(p_cal,   y),
        "Brier_uncal": float(np.mean((p_uncal - y) ** 2)),
        "Brier_cal":   float(np.mean((p_cal   - y) ** 2)),
        "n_pos":     int(y.sum()),
        "n_neg":     int(len(y) - y.sum()),
    })

cal_compare = pd.DataFrame(rows).set_index("outcome")
cal_compare["dECE"]   = cal_compare["ECE_cal"]   - cal_compare["ECE_uncal"]
cal_compare["dNLL"]   = cal_compare["NLL_cal"]   - cal_compare["NLL_uncal"]
cal_compare["dBrier"] = cal_compare["Brier_cal"] - cal_compare["Brier_uncal"]
print("\nBefore/after temperature scaling (T fit on val, evaluated on test).")
print("Rank metrics are unchanged by monotone T-scaling; NLL/Brier/ECE should drop (negative delta).")
display(cal_compare.round(4))

# Support-weighted aggregate
w = cal_compare["n_pos"].astype(float).values
w = w / w.sum() if w.sum() > 0 else np.ones_like(w) / len(w)
print(f"\n  weighted ECE_uncal = {(cal_compare['ECE_uncal'].values * w).sum():.4f}")
print(f"  weighted ECE_cal   = {(cal_compare['ECE_cal'].values   * w).sum():.4f}")
print(f"  weighted dECE      = {(cal_compare['dECE'].values      * w).sum():+.4f}  (negative = better)")
print(f"  weighted dNLL      = {(cal_compare['dNLL'].values      * w).sum():+.4f}")
print(f"  weighted dBrier    = {(cal_compare['dBrier'].values    * w).sum():+.4f}")


### 3f) Reliability Diagram (uncal vs. cal, with ECE annotation)

Per-outcome reliability curves on the test split, before and after applying the
per-outcome `T_k` fit on val. Each panel's title reports uncal -> cal ECE and
the fitted `T_k`. The figure is saved to `images/ar_reliability.png` for the
appendix.


In [ ]:
def reliability_diagram(ax, probs, labels, n_bins=10, label="", color=None):
    bin_edges = np.linspace(0, 1, n_bins + 1)
    bc, mt = [], []
    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        mask = (probs >= lo) & (probs < hi)
        if mask.sum() > 0:
            bc.append(probs[mask].mean()); mt.append(labels[mask].mean())
    if not bc:
        return
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="Perfect")
    ax.plot(bc, mt, "o-", color=color, label=label)


eligible = [n for n in cal_data if cal_data[n][1].sum() >= 5]
n_plots = min(len(eligible), 8)
if n_plots:
    fig, axes = plt.subplots(2, (n_plots + 1) // 2,
                              figsize=(4 * ((n_plots + 1) // 2), 8))
    axes = np.atleast_1d(axes).flatten()
    for ax, name in zip(axes, eligible[:n_plots]):
        z, y = cal_data[name]
        p_uncal = _sigmoid(z)
        p_cal   = _sigmoid(z / temperatures[name])
        ece_un = _ece(p_uncal, y, n_bins=15)
        ece_ca = _ece(p_cal,   y, n_bins=15)
        title = name.replace("_EVENT", "").replace("_COMPLICATION", "")
        ax.set_title(f"{title}\nECE: {ece_un:.3f} -> {ece_ca:.3f}  (T={temperatures[name]:.2f})", fontsize=9)
        reliability_diagram(ax, p_uncal, y, label="Uncalibrated")
        reliability_diagram(ax, p_cal,   y, label="Calibrated")
        ax.set_xlabel("Mean predicted probability"); ax.set_ylabel("Fraction of positives")
        ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.legend(fontsize=8)
    for ax in axes[n_plots:]:
        ax.set_visible(False)
    plt.tight_layout()
    plt.suptitle("Reliability diagrams (INTERVenE-Ar; T fit on val, evaluated on test)",
                 y=1.01, fontsize=11)
    out_path = IMAGES_DIR / "ar_reliability.png"
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    print(f"Saved reliability figure to {out_path}")
    plt.show()
else:
    print("Not enough positives for reliability diagrams.")


### 3g) Risk-trajectory plot (calibrated, single-panel DEATH exemplar)

Illustrative figure for the AR variant's *additive* deliverable: a per-step
calibrated risk curve over the generated trajectory, an output the encoder does
not produce. We show one TP exemplar (a patient whose model risk starts low,
rises, and peaks at the actual death time) and one TN exemplar (a clear
negative, lowest-peak in the subset). The TP's true death event is marked with
a dotted vertical line.

Selection criteria for the TP (tunable knobs at the top of the next cell):
low baseline at generation start, visible amplitude (peak $-$ baseline), peak
within 24\,h of the GT event, and GT event $\ge 150$\,h from admission so the
climb has room to develop.

Set `FOCUS_OUTCOMES = None` in the next cell to render the full grid
over all outcomes (for supplementary / exploration); the default is a single
DEATH panel.


In [ ]:
risk_curves = pd.read_csv(CALIB_DIR / "risk_curves_test_subset.csv")
gt_events   = pd.read_csv(CALIB_DIR / "gt_events_test_subset.csv")
print(f"[load] risk_curves: {len(risk_curves):>6} rows / "
      f"{risk_curves['PatientId'].nunique():>4} patients")
print(f"[load] gt_events:   {len(gt_events):>6} rows / "
      f"{gt_events['PatientId'].nunique():>4} patients")

INPUT_CUTOFF_HOURS = 48.0
HORIZON_HOURS      = 336.0

# ---------- TP selection thresholds (tier 1 = the visual sweet spot) ----------
TIER1_PROB_START_MAX = 0.30   # low baseline at generation start
TIER1_PEAK_MIN       = 0.50   # still a TP (peak above the 0.5 threshold)
TIER1_AMPLITUDE_MIN  = 0.30   # visible rise (peak - baseline)
TIER1_DIST_MAX_H     = 24.0   # peak coincides with a GT event
TIER1_LATE_EVENT_H   = 150.0  # GT event well into the horizon, room for a climb

# Tier-2 / Tier-3 relaxations
TIER2_AMPLITUDE_MIN  = 0.20
TIER2_DIST_MAX_H     = 48.0

# ---------- Manual PID pins (override the picker) ----------
# Per-outcome forced TP PatientId. Set to {} to use the picker for every outcome.
FORCE_TP_PID = {"DEATH_EVENT": 26275344}

# ---------- What to plot ----------
# Set to a list with one outcome for a single-panel figure (paper appendix);
# set to None for the full 2x3 grid over all outcomes (exploration).
FOCUS_OUTCOMES = ["DEATH_EVENT"]

# -----------------------------------------------------------------------------
# Apply per-outcome T to per-step logits to get calibrated per-step probabilities.
for name in OUTCOMES_CAL:
    z_col = f"logit_{name}"
    if z_col in risk_curves.columns:
        risk_curves[f"P_cal_{name}"] = _sigmoid(
            risk_curves[z_col].values / float(temperatures[name])
        )


def _categorize(pid, outcome, thresh=0.5):
    rec = peaks_test[(peaks_test["PatientId"] == pid) & (peaks_test["outcome"] == outcome)]
    if rec.empty:
        return None, None
    label = int(rec["patient_label"].iloc[0])
    peak  = float(rec["peak_prob"].iloc[0])
    pred  = peak >= thresh
    if pred and label:        return "TP", peak
    if pred and not label:    return "FP", peak
    if not pred and label:    return "FN", peak
    return "TN", peak


def _pick_tp_exemplar(outcome):
    """Pick a TP whose calibrated risk curve starts low, rises clearly, and
    peaks at a late GT event.

    For each candidate TP, within [48h, 336h]:
        prob_start  = P_cal at the earliest generated step
        baseline    = min(P_cal)
        peak        = max(P_cal)
        peak_time   = TimePoint at argmax
        amplitude   = peak - baseline
        dist        = min over GT events of |peak_time - gt_t|
        earliest_gt = min(gt_time_hours)

    Tier 1: prob_start <= 0.30 AND amplitude >= 0.30 AND peak >= 0.50 AND
            dist <= 24h AND earliest_gt >= 150h  ->  argmax amplitude.
    Tier 2: amplitude >= 0.20 AND dist <= 48h  ->  argmax amplitude.
    Tier 3 (fallback): argmin dist regardless.
    """
    pcol = f"P_cal_{outcome}"
    if pcol not in risk_curves.columns:
        return None
    candidates = []
    for pid in risk_curves["PatientId"].unique().tolist():
        cat, peak = _categorize(pid, outcome)
        if cat != "TP" or peak is None:
            continue
        gt = gt_events[(gt_events["PatientId"] == pid) & (gt_events["outcome"] == outcome)]
        if gt.empty:
            continue
        sub = risk_curves[(risk_curves["PatientId"] == pid)
                          & (risk_curves["TimePoint"] >= INPUT_CUTOFF_HOURS)
                          & (risk_curves["TimePoint"] <= HORIZON_HOURS)].sort_values("TimePoint")
        if sub.empty:
            continue
        prob_start  = float(sub[pcol].iloc[0])
        baseline    = float(sub[pcol].min())
        peak_v      = float(sub[pcol].max())
        peak_time   = float(sub.loc[sub[pcol].idxmax(), "TimePoint"])
        amplitude   = peak_v - baseline
        dist        = min(abs(peak_time - float(t)) for t in gt["gt_time_hours"].values)
        earliest_gt = float(gt["gt_time_hours"].min())
        candidates.append({
            "pid":         pid,
            "prob_start":  prob_start,
            "peak":        peak_v,
            "peak_time":   peak_time,
            "amplitude":   amplitude,
            "dist":        dist,
            "earliest_gt": earliest_gt,
        })
    if not candidates:
        return None

    tier1 = [c for c in candidates
             if c["prob_start"] <= TIER1_PROB_START_MAX
             and c["amplitude"] >= TIER1_AMPLITUDE_MIN
             and c["peak"]      >= TIER1_PEAK_MIN
             and c["dist"]      <= TIER1_DIST_MAX_H
             and c["earliest_gt"] >= TIER1_LATE_EVENT_H]
    if tier1:
        chosen = max(tier1, key=lambda c: c["amplitude"])
        print(f"[picker tier1] {outcome}: PID {chosen['pid']}  "
              f"start={chosen['prob_start']:.2f} peak={chosen['peak']:.2f} "
              f"amp={chosen['amplitude']:.2f} dist={chosen['dist']:.1f}h "
              f"gt={chosen['earliest_gt']:.1f}h")
        return chosen["pid"]

    tier2 = [c for c in candidates
             if c["amplitude"] >= TIER2_AMPLITUDE_MIN
             and c["dist"]      <= TIER2_DIST_MAX_H]
    if tier2:
        chosen = max(tier2, key=lambda c: c["amplitude"])
        print(f"[picker tier2] {outcome}: PID {chosen['pid']}  "
              f"start={chosen['prob_start']:.2f} peak={chosen['peak']:.2f} "
              f"amp={chosen['amplitude']:.2f} dist={chosen['dist']:.1f}h "
              f"gt={chosen['earliest_gt']:.1f}h")
        return chosen["pid"]

    chosen = min(candidates, key=lambda c: c["dist"])
    print(f"[picker tier3 fallback] {outcome}: PID {chosen['pid']}  "
          f"start={chosen['prob_start']:.2f} peak={chosen['peak']:.2f} "
          f"amp={chosen['amplitude']:.2f} dist={chosen['dist']:.1f}h "
          f"gt={chosen['earliest_gt']:.1f}h")
    return chosen["pid"]


def _pick_tn_exemplar(outcome):
    """Pick the lowest-peak TN -- clearest negative."""
    cands = []
    for pid in risk_curves["PatientId"].unique().tolist():
        cat, peak = _categorize(pid, outcome)
        if cat == "TN" and peak is not None:
            cands.append((pid, peak))
    if not cands:
        return None
    return min(cands, key=lambda x: x[1])[0]


def _plot_pair(ax, focus_outcome):
    short = focus_outcome.replace("_EVENT", "").replace("_COMPLICATION", "")
    pcol  = f"P_cal_{focus_outcome}"
    if pcol not in risk_curves.columns:
        ax.set_axis_off(); return

    pid_tp = FORCE_TP_PID.get(focus_outcome) or _pick_tp_exemplar(focus_outcome)
    pid_tn = _pick_tn_exemplar(focus_outcome)
    plotted = False

    if pid_tp is not None:
        sub_tp = risk_curves[(risk_curves["PatientId"] == pid_tp)
                             & (risk_curves["TimePoint"] >= INPUT_CUTOFF_HOURS)
                             & (risk_curves["TimePoint"] <= HORIZON_HOURS)].sort_values("TimePoint")
        if not sub_tp.empty:
            ax.plot(sub_tp["TimePoint"], sub_tp[pcol], lw=1.8, color="C3",
                    label=f"TP (PID {pid_tp})")
            gt = gt_events[(gt_events["PatientId"] == pid_tp)
                           & (gt_events["outcome"] == focus_outcome)]
            for j, t in enumerate(gt["gt_time_hours"].values):
                ax.axvline(t, color="C3", linestyle=":", lw=1.2,
                           label="GT event (TP)" if j == 0 else None)
            plotted = True

    if pid_tn is not None:
        sub_tn = risk_curves[(risk_curves["PatientId"] == pid_tn)
                             & (risk_curves["TimePoint"] >= INPUT_CUTOFF_HOURS)
                             & (risk_curves["TimePoint"] <= HORIZON_HOURS)].sort_values("TimePoint")
        if not sub_tn.empty:
            ax.plot(sub_tn["TimePoint"], sub_tn[pcol], lw=1.8, color="C0",
                    label=f"TN (PID {pid_tn})")
            plotted = True

    if not plotted:
        ax.set_axis_off(); return

    # Adaptive y-axis: cap at max of (TP, TN) curves with 10% headroom,
    # so small-magnitude bumps are visible. Floor at 0; minimum upper
    # bound at 0.10 so even tiny curves are readable.
    y_top = 0.0
    if pid_tp is not None and not sub_tp.empty:
        y_top = max(y_top, float(sub_tp[pcol].max()))
    if pid_tn is not None and not sub_tn.empty:
        y_top = max(y_top, float(sub_tn[pcol].max()))
    y_top = max(0.10, y_top * 1.10)

    ax.set_title(short, fontsize=11)
    ax.set_xlabel("Hours from admission")
    ax.set_ylabel("P(outcome)")
    ax.set_ylim(0, y_top)
    ax.set_xlim(INPUT_CUTOFF_HOURS, HORIZON_HOURS)
    ax.legend(fontsize=8, loc="best")


# -----------------------------------------------------------------------------
# Render: single panel if FOCUS_OUTCOMES is a single-element list;
# 2x3 grid over all outcomes otherwise.
if FOCUS_OUTCOMES is not None and len(FOCUS_OUTCOMES) == 1:
    fig, ax = plt.subplots(1, 1, figsize=(6.5, 4.0))
    _plot_pair(ax, FOCUS_OUTCOMES[0])
    plt.tight_layout()
    out_path = IMAGES_DIR / "ar_risk_trajectory_death.png"
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    print(f"Saved single-panel trajectory figure to {out_path}")
    plt.show()
else:
    PREVALENCE_ORDER = [
        "HYPERGLYCEMIA_EVENT", "DEATH_EVENT", "SEVERE_HYPERGLYCEMIA_EVENT",
        "KIDNEY_COMPLICATION_EVENT", "HYPOGLYCEMIA_EVENT", "SEVERE_HYPOGLYCEMIA_EVENT",
    ]
    plot_outcomes = [o for o in PREVALENCE_ORDER if o in OUTCOMES_CAL]
    plot_outcomes += [o for o in OUTCOMES_CAL if o not in plot_outcomes]
    plot_outcomes = plot_outcomes[:6]

    ncols, nrows = 3, 2
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.0 * ncols, 3.5 * nrows), squeeze=False)
    for i, outcome in enumerate(plot_outcomes):
        ax = axes[i // ncols, i % ncols]
        _plot_pair(ax, outcome)
    for j in range(len(plot_outcomes), nrows * ncols):
        axes[j // ncols, j % ncols].set_axis_off()
    plt.tight_layout()
    plt.suptitle("INTERVenE-Ar calibrated risk trajectories -- TP (rising toward GT) vs.\\ TN exemplars",
                 y=1.01, fontsize=11)
    out_path = IMAGES_DIR / "ar_risk_trajectories.png"
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    print(f"Saved grid trajectory figure to {out_path}")